# Instrument and Verify a Voice Agent

Instrument a self-hosted voice agent so Future AGI reads it as a call, then run twelve machine-checked gates that either pass or name exactly which column will be blank.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/main/quickstart/instrument-and-verify-voice.ipynb)

A voice call is not a trace with audio in it. The Voice tab finds a call by six conditions at once, and a span that misses any one of them is invisible there no matter how healthy it looks in Traces.

This notebook is the page's own worked example, unchanged. It needs one model key. No LiveKit account, no room, no phone number and no microphone: `AgentSession.run()` is LiveKit's own harness, and `session.start()` takes no room.


## Install

`traceai-livekit` is the instrumentor for this agent's framework. Swap it for `traceai-pipecat` when the agent runs on Pipecat.

In [ ]:
%pip install "livekit-agents[openai]" fi-instrumentation-otel traceai-livekit --quiet


In [ ]:
import os

os.environ["FI_API_KEY"] = "your-api-key"          # app.futureagi.com -> Keys
os.environ["FI_SECRET_KEY"] = "your-secret-key"
os.environ["FI_PROJECT_NAME"] = "my-voice-agent"
os.environ["LLM_API_KEY"] = "your-model-key"       # the agent's own provider key, never a Future AGI one
os.environ["FI_VERIFY"] = "1"
os.environ["FI_VOICE_NO_RECORDING"] = "1"          # this run keeps no recording, per Step 5


## Get the checker

`fi_verify_voice.py` is one file with no dependency outside the standard library. It is the only thing that decides the result.

In [ ]:
!mkdir -p observability/futureagi
!curl -fsSL https://docs.futureagi.com/fi_verify_voice.py -o observability/futureagi/fi_verify_voice.py
!touch observability/__init__.py observability/futureagi/__init__.py
!shasum -a 256 observability/futureagi/fi_verify_voice.py
# 9e487b4e8eb00c1adfb57e2cfdda182005cb8f99d85e909e6531d7730380be2f


## Step 1: Prove the keys before writing code that depends on them

`preflight` sends one real conversation-shaped span, then three deliberately broken variants. If any broken variant is accepted, nothing is proven and it fails. One call is in the Voice tab before you write a line of agent code.

In [ ]:
!python observability/futureagi/fi_verify_voice.py preflight


## Step 2: Provider, mapper, capture

`enable_http_attribute_mapping()` before anything wraps an exporter, and `attach()` after it. `traceai-livekit` rewrites attributes inside `export()`, so a span processor would report on attributes that were never sent.

In [ ]:
%%writefile observability/futureagi/setup.py
# observability/futureagi/setup.py     Future AGI SDK track
import os

from fi_instrumentation import FITracer, register
from fi_instrumentation.fi_types import ProjectType
from traceai_livekit import enable_http_attribute_mapping

from . import fi_verify_voice, livekit_pii_alias

# Import once at process start, AFTER whatever loads your .env, and before any
# LiveKit import that builds a session. Imported earlier the keys are not there
# yet, and only V1 says so.
provider = register(
    project_name=os.environ["FI_PROJECT_NAME"],
    project_type=ProjectType.OBSERVE,
    set_global_tracer_provider=True,      # LiveKit's own spans need the global provider
)

# 1. swap FI's exporter for the one that maps LiveKit attributes.
enable_http_attribute_mapping()

# 2. put the conversation content back on the keys that mapper reads.
livekit_pii_alias.install(provider)

# 3. capture what was really sent, at the exporter, after the mapping.
if os.getenv("FI_VERIFY", "1") == "1":
    fi_verify_voice.attach(provider)

# FITracer, not get_tracer(): a plain Tracer drops session.id and user.id, failing V5
tracer = FITracer(provider.get_tracer("voice-agent"))


One shim, only on LiveKit. LiveKit Agents moved conversation content behind a `pii` segment and `traceai-livekit` 0.1.1 still reads the bare names, so V11 fails without it. Delete it once the instrumentor reads the prefixed names.

In [ ]:
%%writefile observability/futureagi/livekit_pii_alias.py
# observability/futureagi/livekit_pii_alias.py     Future AGI SDK track
ALIAS = {
    "lk.pii.user_input": "lk.user_input",
    "lk.pii.chat_ctx": "lk.chat_ctx",
    "lk.pii.response.text": "lk.response.text",
    "lk.pii.response.function_calls": "lk.response.function_calls",
    "lk.pii.function_tool.arguments": "lk.function_tool.arguments",
    "lk.pii.function_tool.output": "lk.function_tool.output",
    "lk.pii.input_text": "lk.input_text",
    "lk.pii.instructions": "lk.instructions",
    "lk.pii.room_name": "lk.room_name",
    "lk.pii.user_transcript": "lk.user_transcript",
    "lk.pii.participant_identity": "lk.participant_identity",
}


def install(provider):
    """Wrap every exporter on the provider. Call after enable_http_attribute_mapping()."""
    active = getattr(provider, "_active_span_processor", None)
    procs = list(getattr(active, "_span_processors", ())) or ([active] if active else [])
    for proc in procs:
        exp = getattr(proc, "span_exporter", None) or getattr(proc, "_exporter", None)
        if exp is None or getattr(exp, "_lk_pii_alias", False):
            continue
        real = exp.export

        def export(spans, _real=real):
            for s in spans:
                a = getattr(s, "_attributes", None)
                if not a:
                    continue
                add = {new: a[old] for old, new in ALIAS.items() if old in a and new not in a}
                if add:
                    s._attributes = {**dict(a), **add}
            return _real(spans)

        exp.export = export
        exp._lk_pii_alias = True


## Step 5: Write the attributes behind each column

Nothing about a voice call is derived. Duration, turns, talk ratio and the transcript are all read from named attributes on the conversation span, and the instrumentor writes none of them.

Written here, before Steps 3 and 4, because `agent.py` imports this module.

In [ ]:
%%writefile observability/futureagi/voice_spans.py
# observability/futureagi/voice_spans.py     both tracks
import json, uuid

DURATION = "call.duration"                    # seconds, number. Duration column, duration filter
TURNS = "call.total_turns"                    # number. Turns column, turn_count filter
TALK_RATIO = "call.talk_ratio"                # 0..1, number. Talk ratio filter
STATUS = "call.status"                        # provider status string
PHONE = "call.participant_phone_number"
PROVIDER = "gen_ai.system"                    # which parser the server uses for this call
TRANSCRIPT = "conversation.transcript"        # the whole thing, as JSON. What voice evals bind to
TRANSCRIPT_RENDERED = "fi.conversation.transcript"   # the same list. What the call drawer renders
RECORDING_MONO = "conversation.recording.mono.combined"
RECORDING_STEREO = "conversation.recording.stereo"

AGENT_ROLES = ("assistant", "agent", "bot")
CALLER_ROLES = ("user", "customer", "caller")


def _rows(turns):
    """turns is [(role, text), ...] in order, or [(role, text, start, duration), ...]
    where start is seconds from the beginning of the call and duration is how long
    that utterance took to speak."""
    return [tuple(t) + (None,) * (4 - len(t)) for t in turns]


def write_transcript(span, turns):
    """THREE keys, because three surfaces read it and none of them falls back to
    another. Write two of the three and the call looks complete on one screen and
    empty on the next.

      fi.conversation.transcript          the call drawer renders this one, and
                                          only this one, on a self-hosted agent
      conversation.transcript             what the eval variable picker resolves
      conversation.transcript.N.message.* what the error feed and the I/O panels walk

    The per-turn start and duration are what the Call Analytics strip computes
    Duration, Latency, User / AI and Silence from. Leave them out and those four
    cards read blank while Turns and Words are still right.
    """
    turns = _rows(turns)
    span.set_attribute(TRANSCRIPT_RENDERED, json.dumps(
        [{"id": str(uuid.uuid4()), "role": r, "content": c,
          "time": None if s is None else str(s), "duration": d}
         for r, c, s, d in turns]))
    span.set_attribute(TRANSCRIPT, json.dumps(
        [{"role": r, "content": c} for r, c, _, _ in turns]))
    for i, (role, text, _, _) in enumerate(turns):
        span.set_attribute("conversation.transcript.%d.message.role" % i, role)
        span.set_attribute("conversation.transcript.%d.message.content" % i, text)


def talk_ratio(turns):
    """Agent share of the words spoken. In an audio deployment use talk TIME."""
    turns = _rows(turns)
    agent = sum(len(c.split()) for r, c, _, _ in turns if r in AGENT_ROLES)
    total = sum(len(c.split()) for _, c, _, _ in turns) or 1
    return round(agent / total, 3)


def finish(span, *, turns, duration, provider, status="completed",
           phone=None, recording=None, stereo=None):
    """Close the conversation span with everything the Voice tab reads."""
    rows = _rows(turns)
    write_transcript(span, rows)
    span.set_attribute(TURNS, len(rows))
    span.set_attribute(DURATION, round(duration, 3))
    span.set_attribute(TALK_RATIO, talk_ratio(rows))
    span.set_attribute(PROVIDER, provider)
    span.set_attribute(STATUS, status)
    if phone:
        span.set_attribute(PHONE, phone)
    if recording:
        span.set_attribute(RECORDING_MONO, recording)
    if stereo:
        span.set_attribute(RECORDING_STEREO, stereo)
    span.set_attribute("input.value",
                       next((c for r, c, _, _ in rows if r in CALLER_ROLES), ""))
    span.set_attribute("output.value",
                       next((c for r, c, _, _ in reversed(rows) if r in AGENT_ROLES), ""))


## Steps 3, 4 and 6: One call, one conversation span, opened early

**Step 3 is the one that decides whether the call exists.** The conversation span has to open *before* `session.start()`. Opened after it, LiveKit's own `agent_session` span is already the root, this span becomes a child, and the Voice tab never lists the call, because it selects a conversation span with no parent. Nothing errors and no other surface reads differently.

In [ ]:
%%writefile agent.py
# agent.py
"""One call, one conversation span, no room and no phone number.

AgentSession.run() is LiveKit's own harness: it drives a real session with no
room, no LiveKit account and no telephony, so this file is the whole worked
example and anyone can run it.
"""
import asyncio, os, sys, time, uuid

from observability.futureagi.setup import provider, tracer      # first import, before livekit
from observability.futureagi import voice_spans

from fi_instrumentation import using_attributes
from livekit.agents import Agent, AgentSession
from livekit.plugins import openai


class Assistant(Agent):
    def __init__(self):
        super().__init__(instructions=(
            "You are a voice assistant for an airline. Answer in one short spoken "
            "sentence, and never read out a list."))


async def main():
    # A call is a conversation, so the example is two exchanges, not one. Anything
    # you pass on the command line replaces them.
    asks = sys.argv[1:] or ["How much baggage can I bring?",
                            "And is a stroller counted separately?"]
    session_id = "call_" + uuid.uuid4().hex[:12]
    user_id = os.getenv("CALLER_ID", "acct_10427")

    base = os.getenv("OPENAI_BASE_URL", "https://api.groq.com/openai/v1")
    key = os.environ["LLM_API_KEY"]
    session = AgentSession(
        stt=openai.STT(model="whisper-large-v3-turbo", base_url=base, api_key=key),
        llm=openai.LLM(model=os.getenv("AGENT_MODEL", "openai/gpt-oss-120b"),
                       base_url=base, api_key=key),
    )

    started = time.monotonic()
    # The conversation span opens BEFORE session.start(). Opened after it, LiveKit's
    # own agent_session span is already the root, this one becomes a child, and the
    # Voice tab never lists the call: it selects a conversation span with no parent.
    with using_attributes(session_id=session_id, user_id=user_id, tags=["prod"]):
        with tracer.start_as_current_span("voice.call", fi_span_kind="conversation") as call:
            await session.start(agent=Assistant())
            for ask in asks:
                await session.run(user_input=ask, input_modality="text")
            turns = [(m.role, m.text_content) for m in session.history.items
                     if getattr(m, "role", None) in ("user", "assistant")
                     and getattr(m, "text_content", None)]
            voice_spans.finish(call, turns=turns,
                               duration=time.monotonic() - started,
                               provider="livekit")
            await session.aclose()

    provider.force_flush()
    for role, text in turns:
        print("  %-9s %s" % (role, text[:100]))
    print("\n  session.id = " + session_id)


if __name__ == "__main__":
    asyncio.run(main())


## Step 6: Run one call and let the checker decide

`check` exits `0`, or it names the gate that failed and why. Pass your own asks as arguments to `agent.py` to replace the two below.

In [ ]:
!python agent.py


In [ ]:
!python observability/futureagi/fi_verify_voice.py check


## What GREEN means

Twelve gates, and the one that tells a working integration from a silently invisible one is **V3**: a conversation-typed span with no parent. Eleven of the twelve still pass on a call the Voice tab will never list.

Open the project's Voice tab and the call is there, with its transcript, turn count and word count read straight back off the attributes Step 5 wrote.

The full page, including the OpenTelemetry track, the eval-binding table and troubleshooting: [Instrument and Verify a Voice Agent](https://docs.futureagi.com/docs/cookbook/quickstart/instrument-and-verify-voice).